In [1]:
!pip install pynini
!pip install wurlitzer

In [6]:
from pynini import *
from pynini.lib import rewrite
import itertools
import os

# Functions for Loading the Alphabet

In [ ]:
ko_sym = pn.SymbolTable()
en_sym = pn.SymbolTable()
    
def letter_mappings(txt_file: str):
    mappings = {}
    
    with open(txt_file, "r") as file:
        maps = file.readlines()
        
    for pairs in maps:
        if len(pairs) > 1:
          if not pairs.startswith("#"):
            pairs = pairs.strip()
            pair = pairs.split(" ")
            korean = pair[0]
            english = pair[1]
            mappings[korean] = english
            ko_sym.add_symbol(korean)
            en_sym.add_symbol(english)
              
    return mappings
          
def ccv_mapping(v_maps, c_maps, end_maps):
    ccv_mappings = {}
    
    for c1, ko_c1 in end_maps.items():
        for c2, ko_c2 in c_maps.items():
            for v, ko_V in v_maps.items():
                en = c1+c2+v
                ko = ko_c1+ko_c2+ko_v
                ccv_mappings[ko] = en
                ko_sym.add_symbol(ko)
                en_sym.add_symbol(en)
                
    return ccv_mappings

def rule_mappings(rule_maps, v_maps):
    mappings = {}
    
    for en_let, ko_let in rule_maps.items():
        ko_let = rule_maps
        for en_v, ko_v in v_maps.items():
            en = en_let+en_v
            ko = ko_let+ko_v
            mappings[ko] = en
            ko_sym.add_symbol(ko)
            en_sym.add_symbol(en)
              
    return mappings

## Create the alphabet dictionaries

In [ ]:
# add alphabet

path = "mappings/

endings = path+"ending_consonants_mapping.txt"
endings = add_mappings(endings)

vowels = path+"vowel_mappings.txt"
vowels = add_mappings(vowels)

consonants = path+"consonant_mappings.txt"
consonants = add_mappings(consonants)

ccvs = ccv_mappings(vowels, consonants, endings)

palatizations = path+"palatization_mappings.txt"
palatizations = add_mappings(palatizations)

nasalizations = path + "nasalization_mappings.txt"
nasalizations = add_mappings(nasalizations)

liaisons = path + "liaison_mappings.txt"
liaisons = add_mappings(nasalizations)

# CCV maps with the three different rules:
vowels_palate = {"ㅣ":"i"}
palatizations_ccv = rule_mappings(palatizations, vowels_palate)
nasalizations_ccv = rule_mappings(nasalizations, vowels)
liaisons_ccv = rule_mappings(liaisons, vowels)

In [ ]:
blah = ["ㄱ.","ㅋ.","ㄲ.","ㄺ.","ㄳ."] + ["ㄷ.","ㄸ.","ㅌ.","ㅅ.","ㅆ.","ㅈ.","ㅉ.","ㅊ.","ㅎ."] + ["ㅂ.","ㅍ.","ㄼ.","ㄿ.","ㅄ."]
nasals_causers = "ㅁㄹㄴ"
non_nasalizations = {}
for a in blah:
  a = a.strip(".")[0]
  for k, e in consonant_map.items():
    if k not in nasals_causers and k != a:
      non_nasalizations[a+k] = consonant_map[a]+consonant_map[k]
      en_sym.add_symbol(non_nasalizations[a+k])
print(non_nasalizations)

## Korean text preprocessing

In [8]:
from jamo import h2j, j2hcj

def preprocess(word: str) -> str:
  '''splits korean word into its alphabet
     adds distinction between syllable words
  '''
  preprocessed_word = ""
  split = list(word)
  for block in split:
    chars = (j2hcj(h2j(block)))
    for c in range(len(chars)-1):
      preprocessed_word += chars[c] + " "
    preprocessed_word += chars[-1] + "." + " "
  return preprocessed_word+"*"

# test
word = "국물"
print(preprocess(word))

ㄱ ㅜ ㄱ. ㅁ ㅜ ㄹ. *


# FST FOR VARIOUS BATCHIM RULES


## Nasalization Rule
Recognizes and translates only words that contain  syllable blocks that need to follow a nasalization rule and only looks at those rules






































In [ ]:
nasalization_FST = pn.Fst()

q0=nasalization_FST.add_state()
q1=nasalization_FST.add_state()
q2=nasalization_FST.add_state()
q3=nasalization_FST.add_state()
q4=nasalization_FST.add_state()
q5=nasalization_FST.add_state()
q6=nasalization_FST.add_state()
q7=nasalization_FST.add_state()
q8=nasalization_FST.add_state()
q9=nasalization_FST.add_state()
q10=nasalization_FST.add_state()
q11=nasalization_FST.add_state()
q12=nasalization_FST.add_state()
q13=nasalization_FST.add_state()
q14=nasalization_FST.add_state()
q15=nasalization_FST.add_state()
q16=nasalization_FST.add_state()
q17=nasalization_FST.add_state()
q18=nasalization_FST.add_state()
q19=nasalization_FST.add_state()
q20=nasalization_FST.add_state()

nasalization_FST.set_start(q0)
nasalization_FST.set_final(q2)

### Add States

In [ ]:
def get_nasal_sounds(symbol):
  sounds = []
  symbol = symbol.strip(".")[0]
  for i in "ㅁㄹㄴ":
    pair = symbol+i
    if pair in nasalization_map:
      sounds.append({"symbol":i, "sound":nasalization_map[pair]})
  return sounds

In [ ]:
en_sym.add_symbol("<eps>", 0)     # epsilon must be 0
en_sym.add_symbol(" ")     # epsilon must be 0
ko_sym.add_symbol("<eps>", 0)     # epsilon must be 0
ko_sym.add_symbol(" ")     # epsilon must be 0

# spaces are used to separate characters, include them in transitions
nasalization_FST.add_arc(q1, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q1))
nasalization_FST.add_arc(q2, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q2))
nasalization_FST.add_arc(q3, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q3))
nasalization_FST.add_arc(q4, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q4))
nasalization_FST.add_arc(q5, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q5))
nasalization_FST.add_arc(q6, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q6))
nasalization_FST.add_arc(q7, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q7))
nasalization_FST.add_arc(q8, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q8))
nasalization_FST.add_arc(q9, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q9))
nasalization_FST.add_arc(q10, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q10))
nasalization_FST.add_arc(q11, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q11))
nasalization_FST.add_arc(q12, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q12))
nasalization_FST.add_arc(q13, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q13))
nasalization_FST.add_arc(q14, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q14))
nasalization_FST.add_arc(q15, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q15))
nasalization_FST.add_arc(q16, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q16))
nasalization_FST.add_arc(q17, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q17))
nasalization_FST.add_arc(q18, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q18))
nasalization_FST.add_arc(q19, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q19))
nasalization_FST.add_arc(q20, pn.Arc(ko_sym.find(" "), ko_sym.find(" "), 0, q20))

# q0 -> q0 move for any consonant letter
for ck, cen in consonant_map.items():
  nasalization_FST.add_arc(q0, pn.Arc(ko_sym.find(ck), en_sym.find(cen), 0, q1))

# q1 -> q1 stay for any vowel
for vk, ve in vowel_map.items():
  ve_num = en_sym.find(ve)
  vk_num = ko_sym.find(vk)
  nasalization_FST.add_arc(q1, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q2, pn.Arc(vk_num, ve_num, 0, q2))
  # from other states to vowel
  nasalization_FST.add_arc(q9,  pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q10, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q11, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q12, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q13, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q14, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q15, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q16, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q17, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q18, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q19, pn.Arc(vk_num, ve_num, 0, q2))
  nasalization_FST.add_arc(q20, pn.Arc(vk_num, ve_num, 0, q2))

'''
For other consonants after vowel: look for nasalizations
1. all characters that follow nasalization rule should be mapped empty string
2. the next characters after should be either the nasalization (romanized) or the regular (romanized)
'''
# q1 -> q2, q3, or q4 if nasal ending
# q1 -> q1 if any other ending
nasal_endings_g = ["ㄱ.","ㅋ.","ㄲ.","ㄺ.","ㄳ."]
nasal_endings_d = ["ㄷ.","ㄸ.","ㅌ.","ㅅ.","ㅆ.","ㅈ.","ㅉ.","ㅊ.","ㅎ."]
nasal_endings_b = ["ㅂ.","ㅍ.","ㄼ.","ㄿ.","ㅄ."]

for ck, ce in ending_consonant_map.items():
  ck_end = ko_sym.find(ck)
  ce_end = en_sym.find(ce)

  if ck in nasal_endings_g:
    nasalization_FST.add_arc(q2, pn.Arc(ck_end, 0, 0, q3)) # ㄱ. -> g
    sounds = get_nasal_sounds(ck)
    for i in sounds:
      sound = en_sym.find(i["sound"])
      symbol = ko_sym.find(i["symbol"])
      nasalization_FST.add_arc(q3, pn.Arc(symbol, sound, 0, q9))

  elif ck in nasal_endings_d:
    nasalization_FST.add_arc(q2, pn.Arc(ck_end, 0, 0, q4))
    sounds = get_nasal_sounds(ck)
    for i in sounds:
      sound = en_sym.find(i["sound"])
      symbol = ko_sym.find(i["symbol"])
      nasalization_FST.add_arc(q4, pn.Arc(symbol, sound, 0, q11))

  elif ck in nasal_endings_b:
    nasalization_FST.add_arc(q2, pn.Arc(ck_end, 0, 0, q5))
    sounds = get_nasal_sounds(ck)
    for i in sounds:
      sound = en_sym.find(i["sound"])
      symbol = ko_sym.find(i["symbol"])
      nasalization_FST.add_arc(q5, pn.Arc(symbol, sound, 0, q13))

  elif ck == "ㅇ.":
    nasalization_FST.add_arc(q2, pn.Arc(ck_end, 0, 0, q6))
    sounds = get_nasal_sounds(ck)
    for i in sounds:
      sound = en_sym.find(i["sound"])
      symbol = ko_sym.find(i["symbol"])
      nasalization_FST.add_arc(q6, pn.Arc(symbol, sound, 0, q15))

  elif ck == "ㄴ.":
    nasalization_FST.add_arc(q2, pn.Arc(ck_end, 0, 0, q7))
    sounds = get_nasal_sounds(ck)
    for i in sounds:
      sound = en_sym.find(i["sound"])
      symbol = ko_sym.find(i["symbol"])
      nasalization_FST.add_arc(q7, pn.Arc(symbol, sound, 0, q17))

  elif ck == "ㅁ.":
    nasalization_FST.add_arc(q2, pn.Arc(ck_end, 0, 0, q8))
    sounds = get_nasal_sounds(ck)
    for i in sounds:
      sound = en_sym.find(i["sound"])
      symbol = ko_sym.find(i["symbol"])
      nasalization_FST.add_arc(q8, pn.Arc(symbol, sound, 0, q19))

  else:
    nasalization_FST.add_arc(q2, pn.Arc(ck_end, ce_end, 0, q2))

# add path non-nasalization chars
for k, e in non_nasalizations.items():
  let = k[0]+"."
  next_char = k[1]
  if let in nasal_endings_g:
    nasalization_FST.add_arc(q3,  pn.Arc(ko_sym.find(next_char), en_sym.find(e), 0, q10))

  elif let in nasal_endings_d:
    nasalization_FST.add_arc(q4,  pn.Arc(ko_sym.find(next_char), en_sym.find(e), 0, q12))

  elif let in nasal_endings_b:
    nasalization_FST.add_arc(q5,  pn.Arc(ko_sym.find(next_char), en_sym.find(e), 0, q14))

  elif let == "ㅇ.":
    nasalization_FST.add_arc(q6,  pn.Arc(ko_sym.find(next_char), en_sym.find(e), 0, q16))

  elif let == "ㄴ.":
    nasalization_FST.add_arc(q7,  pn.Arc(ko_sym.find(next_char), en_sym.find(e), 0, q18))

  elif let == "ㅁ.":
    nasalization_FST.add_arc(q8,  pn.Arc(ko_sym.find(next_char), en_sym.find(e), 0, q20))


In [ ]:
print(nasalization_FST.start())
print(nasalization_FST.num_states())
print(nasalization_FST.verify())

In [ ]:
def build_test(korean: list):
    test_fst = pn.Fst()
    first = test_fst.add_state()
    test_fst.set_start(first)

    for ko in korean:
      next_state = test_fst.add_state()
      test_fst.add_arc(first, pn.Arc(ko_sym.find(ko), ko_sym.find(ko), 0, next_state))
      first = next_state

    test_fst.set_final(first)
    test_fst.set_input_symbols(ko_sym)
    test_fst.set_output_symbols(ko_sym)
    print(test_fst.start())
    print(test_fst.num_states())
    print(test_fst.verify())
    return test_fst

In [ ]:
nasalization_FST.set_input_symbols(ko_sym)
nasalization_FST.set_output_symbols(en_sym)
preprocessed = preprocess("국물")
lst = ['ㄱ', ' ', 'ㅜ', ' ', 'ㄱ.', ' ', 'ㅁ', ' ', 'ㅜ', ' ', 'ㄹ.', ' ']
test_fst = build_test(lst)

lattice = pn.compose(test_fst, nasalization_FST)

best = pn.shortestpath(lattice, nshortest=1)
print(best.string(token_type=en_sym))

In [ ]:
test_fst

In [ ]:
# nasalization_FST

In [ ]:
for state in range(test_fst.num_states()):
  for arc in test_fst.arcs(state):
    print(state, "->", arc.nextstate,
                "input:", arc.ilabel,
                "output:", arc.olabel,
                "weight:", arc.weight)

In [ ]:
ko_sym.find(" ")
en_sym.find(" ")

In [ ]:
for state in range(nasalization_FST.num_states()):
  for arc in nasalization_FST.arcs(state):
    print(state, "->", arc.nextstate,
                "input:", arc.ilabel,
                "output:", arc.olabel,
                "weight:", arc.weight)